In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import warnings
import os
import zipfile
import gc
import itertools
from google.colab import files

warnings.filterwarnings("ignore")

output_dir = "h3_simulation_results"
os.makedirs(output_dir, exist_ok=True)

# メモリクラッシュ回避のためのセーフモード設定
N_list = [1000, 10000, 100000, 1000000, 2500000]
D = 15
T = 5

results_summary = []

print("Starting H3 Macro Simulation: Micro-Cell Divergence via Interaction Terms...")
print(f"Base Dimensions (D) = {D}, Total Features (K) = {D + (D*(D-1))//2}")

for N in N_list:
    print(f"\n--- Running for N = {N:,} ---")
    trial_results = []

    for t in range(T):
        y = np.random.binomial(1, 0.5, N).astype(np.int8)
        p_null = np.mean(y)
        ll_null = -N * log_loss(y, np.full(N, p_null))

        p_j = np.random.uniform(0.01, 0.2, D)
        X_main = np.random.binomial(1, p_j, size=(N, D)).astype(np.int8)

        clf_main = LogisticRegression(penalty=None, solver='lbfgs', max_iter=100, n_jobs=-1)
        clf_main.fit(X_main, y)
        ll_main = -N * log_loss(y, clf_main.predict_proba(X_main))
        delta_ll_main = ll_main - ll_null
        max_beta_main = np.max(np.abs(clf_main.coef_[0]))

        idx_pairs = list(itertools.combinations(range(D), 2))
        num_inter = len(idx_pairs)
        K = D + num_inter

        X_inter = np.zeros((N, num_inter), dtype=np.int8)
        for idx, (i, j) in enumerate(idx_pairs):
            X_inter[:, idx] = X_main[:, i] * X_main[:, j]

        X_full = np.hstack([X_main, X_inter])

        clf_full = LogisticRegression(penalty=None, solver='lbfgs', max_iter=100, n_jobs=-1)
        clf_full.fit(X_full, y)
        ll_full = -N * log_loss(y, clf_full.predict_proba(X_full))
        delta_ll_full = ll_full - ll_null
        max_beta_full = np.max(np.abs(clf_full.coef_[0]))

        del X_main, X_inter, X_full
        gc.collect()

        aic_null = 2 * 1 - 2 * ll_null
        aic_main = 2 * D - 2 * ll_main
        aic_full = 2 * K - 2 * ll_full

        trial_results.append({
            'Trial': t + 1,
            'N': N,
            'Delta_LL_Main': delta_ll_main,
            'Delta_LL_Full': delta_ll_full,
            'Max_Beta_Main': max_beta_main,
            'Max_Beta_Full': max_beta_full,
            'AIC_Drop_Main': aic_null - aic_main,
            'AIC_Drop_Full': aic_null - aic_full
        })

    df_trials = pd.DataFrame(trial_results)
    df_trials.to_csv(f"{output_dir}/h3_trials_N_{N}.csv", index=False)

    mean_ll_main = df_trials['Delta_LL_Main'].mean()
    mean_ll_full = df_trials['Delta_LL_Full'].mean()
    mcse_ll_full = df_trials['Delta_LL_Full'].std() / np.sqrt(T)

    mean_beta_main = df_trials['Max_Beta_Main'].mean()
    mean_beta_full = df_trials['Max_Beta_Full'].mean()

    print(f"  Theoretical Delta LL: Main (D/2)={D/2}, Full (K/2)={K/2}")
    print(f"  Empirical Delta LL  : Main={mean_ll_main:.2f}, Full={mean_ll_full:.2f} (MCSE: {mcse_ll_full:.4f})")
    print(f"  Max |Beta|          : Main={mean_beta_main:.2f}, Full={mean_beta_full:.2f}")

    results_summary.append({
        'Sample_Size_N': N,
        'Mean_Delta_LL_Main': mean_ll_main,
        'Mean_Delta_LL_Full': mean_ll_full,
        'MCSE_Delta_LL_Full': mcse_ll_full,
        'Mean_Max_Beta_Main': mean_beta_main,
        'Mean_Max_Beta_Full': mean_beta_full
    })

df_summary = pd.DataFrame(results_summary)
df_summary.to_csv(f"{output_dir}/h3_summary_metrics.csv", index=False)

plt.style.use('seaborn-v0_8-whitegrid')

plt.figure(figsize=(10, 6))
plt.plot(df_summary['Sample_Size_N'], df_summary['Mean_Delta_LL_Full'],
         marker='o', color='darkred', linewidth=2, label=f'Main + Interactions (K={K})')
plt.plot(df_summary['Sample_Size_N'], df_summary['Mean_Delta_LL_Main'],
         marker='s', color='darkorange', linewidth=2, label=f'Main Effects Only (D={D})')

plt.axhline(y=K/2, color='gray', linestyle='--', linewidth=2, label=f'Theoretical Expectation for Full (K/2 = {K/2})')
plt.axhline(y=D/2, color='gray', linestyle=':', linewidth=2, label=f'Theoretical Expectation for Main (D/2 = {D/2})')

plt.fill_between(df_summary['Sample_Size_N'],
                 df_summary['Mean_Delta_LL_Full'] - 1.96 * df_summary['MCSE_Delta_LL_Full'],
                 df_summary['Mean_Delta_LL_Full'] + 1.96 * df_summary['MCSE_Delta_LL_Full'],
                 color='darkred', alpha=0.2, label='95% CI (MCSE)')

plt.xscale('log')
plt.xlabel('Sample Size N (log scale)', fontsize=12)
plt.ylabel('Log-Likelihood Inflation ($\Delta LL$)', fontsize=12)
plt.title('H3: Likelihood Divergence via Combinatorial Micro-Cells', fontsize=14)
plt.legend(fontsize=10)
plt.savefig(f"{output_dir}/fig1_likelihood_divergence.png", dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(df_summary['Sample_Size_N'], df_summary['Mean_Max_Beta_Full'],
         marker='D', color='purple', linewidth=2, label='Max |Beta| (Main + Interactions)')
plt.plot(df_summary['Sample_Size_N'], df_summary['Mean_Max_Beta_Main'],
         marker='^', color='orange', linewidth=2, label='Max |Beta| (Main Only)')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Sample Size N (log scale)', fontsize=12)
plt.ylabel(r'Maximum Absolute Coefficient $\max |\hat{\beta}|$ (log scale)', fontsize=12)
plt.title('H3: Complete Separation and Coefficient Explosion', fontsize=14)
plt.legend(fontsize=10)
plt.savefig(f"{output_dir}/fig2_complete_separation.png", dpi=300, bbox_inches='tight')
plt.show()

zip_filename = "H3_Simulation_Results.zip"
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for root, dirs, files_list in os.walk(output_dir):
        for file in files_list:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"\nAll calculations completed successfully. Generating {zip_filename}...")
files.download(zip_filename)

<>:125: SyntaxWarning: invalid escape sequence '\D'
<>:125: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_3634/749466042.py:125: SyntaxWarning: invalid escape sequence '\D'
  plt.ylabel('Log-Likelihood Inflation ($\Delta LL$)', fontsize=12)


Starting H3 Macro Simulation: Micro-Cell Divergence via Interaction Terms...
Base Dimensions (D) = 15, Total Features (K) = 120

--- Running for N = 1,000 ---
  Theoretical Delta LL: Main (D/2)=7.5, Full (K/2)=60.0
  Empirical Delta LL  : Main=8.63, Full=77.98 (MCSE: 1.4167)
  Max |Beta|          : Main=0.63, Full=21.58

--- Running for N = 10,000 ---
  Theoretical Delta LL: Main (D/2)=7.5, Full (K/2)=60.0
  Empirical Delta LL  : Main=6.90, Full=62.58 (MCSE: 3.0992)
  Max |Beta|          : Main=0.23, Full=1.24

--- Running for N = 100,000 ---
  Theoretical Delta LL: Main (D/2)=7.5, Full (K/2)=60.0
  Empirical Delta LL  : Main=7.38, Full=56.76 (MCSE: 2.8152)
  Max |Beta|          : Main=0.05, Full=0.26

--- Running for N = 1,000,000 ---
